# EcoSort: Model Evaluation & Results Analysis
## 12 Municipal Material Classes → 4 Solid Waste Management (SWM) 2026 Streams

This notebook presents empirical evaluation on the held-out **2,843 test images**.
All metrics and figures are loaded directly from generated artifacts:
- `ml/artifacts/wastenet/` (Custom 5-Block CNN from scratch)
- `ml/artifacts/mobilenet_v3/` (MobileNetV3-Large transfer learning)
- `ml/artifacts/efficientnet_b0/` (EfficientNet-B0 transfer learning)
- `ml/reports/figures/` (Confusion matrices, Reliability diagrams, and Grad-CAM grids)


In [ ]:
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image as IPImage, display, Markdown

# Define paths
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
ARTIFACTS_DIR = ROOT / 'ml' / 'artifacts'
REPORTS_DIR = ROOT / 'ml' / 'reports'
FIGURES_DIR = REPORTS_DIR / 'figures'

print('Artifacts Directory:', ARTIFACTS_DIR)
print('Figures Directory:  ', FIGURES_DIR)


## 1. Overall Performance Comparison on Unseen Test Split

Evaluation conducted on 2,843 held-out test images across all three models.
Per Decision **D-16**, MobileNetV3-Large was pre-selected as the serving model prior to test evaluation based on validation macro-F1.


In [ ]:
with open(REPORTS_DIR / 'model_comparison.json') as f:
    comp = json.load(f)

models = ['wastenet', 'mobilenet_v3', 'efficientnet_b0']
names = ['WasteNet (Scratch)', 'MobileNetV3-Large (Transfer)', 'EfficientNet-B0 (Transfer)']

rows = []
for m, name in zip(models, names):
    data = comp['models'][m]
    rows.append({
        'Model': name,
        'Test Acc': f"{data['accuracy']*100:.2f}%",
        'Macro-F1': f"{data['macro_f1']:.4f}",
        '95% CI (Acc)': f"[{data['bootstrap_ci_95']['accuracy']['low']*100:.1f}%, {data['bootstrap_ci_95']['accuracy']['high']*100:.1f}%]",
        'Stream Acc': f"{data['stream_accuracy']*100:.2f}%",
        'Hazard Miss Rate': f"{data['hazard_miss_rate']*100:.2f}% ({data['hazard_miss_count']}/{data['hazard_total']})",
        'Coverage @ Thresh': f"{data['coverage_at_threshold']*100:.1f}% (thr={data['uncertain_threshold']:.2f})",
        'Acc @ Thresh': f"{data['accuracy_above_threshold']*100:.2f}%",
        'CPU Latency p50': f"{data['cpu_latency_ms_p50']:.1f} ms",
        'Params': f"{data['params']:,}",
        'Size': f"{data['size_mb']:.1f} MB",
    })

df_summary = pd.DataFrame(rows)
display(df_summary)


## 2. Statistical Significance: McNemar's Test

We conduct McNemar's test with continuity correction and exact two-tailed binomial p-values on paired binary test predictions ($N=2,843$).


In [ ]:
mc = comp['mcnemar_tests']
print('=== WasteNet vs. MobileNetV3-Large ===')
w_vs_m = mc['wastenet_vs_mobilenet_v3']
print('Contingency Table:', w_vs_m['contingency_table'])
print(f"Discordant pairs: {w_vs_m['discordant_pairs']}")
print(f"Chi-Square:       {w_vs_m['chi2_statistic']:.4f}")
print(f"Exact p-value:    {w_vs_m['exact_p_value']:.4e} (Significant: {w_vs_m['statistically_significant']})")

print('\n=== MobileNetV3-Large vs. EfficientNet-B0 ===')
m_vs_e = mc['mobilenet_v3_vs_efficientnet_b0']
print('Contingency Table:', m_vs_e['contingency_table'])
print(f"Discordant pairs: {m_vs_e['discordant_pairs']}")
print(f"Chi-Square:       {m_vs_e['chi2_statistic']:.4f}")
print(f"Exact p-value:    {m_vs_e['exact_p_value']:.4e} (Significant: {m_vs_e['statistically_significant']})")


## 3. Confusion Matrix Heatmaps (12 Material Classes)

Comparing classification patterns across all three architectures.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6), dpi=150)
for ax, m, name in zip(axes, models, names):
    cm_data = json.load(open(ARTIFACTS_DIR / m / 'confusion_matrix.json'))
    cm = np.array(cm_data['matrix'])
    cm_norm = cm.astype('float') / np.maximum(cm.sum(axis=1)[:, np.newaxis], 1)
    cax = ax.imshow(cm_norm, cmap='Blues')
    ax.set_title(f"{name}\nAcc: {comp['models'][m]['accuracy']*100:.1f}%", fontsize=11, fontweight='bold')
    ax.set_xticks(range(12))
    ax.set_xticklabels(cm_data['classes'], rotation=45, ha='right', fontsize=7)
    ax.set_yticks(range(12))
    ax.set_yticklabels(cm_data['classes'], fontsize=7)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
plt.tight_layout()
plt.show()


## 4. Per-Class F1 Breakdown & Top Material Confusions


In [ ]:
classes = list(comp['models']['mobilenet_v3']['per_class'].keys())
f1_w = [comp['models']['wastenet']['per_class'][c]['f1'] for c in classes]
f1_m = [comp['models']['mobilenet_v3']['per_class'][c]['f1'] for c in classes]
f1_e = [comp['models']['efficientnet_b0']['per_class'][c]['f1'] for c in classes]

x = np.arange(len(classes))
width = 0.28

plt.figure(figsize=(14, 5), dpi=150)
plt.bar(x - width, f1_w, width, label='WasteNet (Scratch)', color='#718096')
plt.bar(x, f1_m, width, label='MobileNetV3 (Default)', color='#2B6CB0')
plt.bar(x + width, f1_e, width, label='EfficientNet-B0', color='#319795')
plt.xticks(x, classes, rotation=35, ha='right', fontweight='bold')
plt.ylabel('F1-Score', fontweight='bold')
plt.title('Per-Class F1-Score Comparison across 12 Municipal Waste Classes', fontweight='bold')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()


## 5. Domain Shift & Per-Source Breakdown

We inspect performance across the 4 aggregated constituent datasets:
- `garbage12`: Curated web-scraped waste image benchmark (Kaggle)
- `realwaste`: Landfill and sorting facility photos under uneven real-world lighting
- `trashbox`: Web-scraped e-waste, hazardous, and municipal items featuring stock watermarks (e.g. Shutterstock, Getty) and varied camera angles
- `trashnet`: Classic laboratory-style photos on uniform white background


In [ ]:
sources = list(comp['models']['mobilenet_v3']['per_source_accuracy'].keys())
src_rows = []
for s in sources:
    src_rows.append({
        'Source Dataset': s,
        'Test Count': comp['models']['mobilenet_v3']['per_source_accuracy'][s]['total'],
        'WasteNet (Scratch)': f"{comp['models']['wastenet']['per_source_accuracy'][s]['accuracy']*100:.1f}%",
        'MobileNetV3 (Default)': f"{comp['models']['mobilenet_v3']['per_source_accuracy'][s]['accuracy']*100:.1f}%",
        'EfficientNet-B0': f"{comp['models']['efficientnet_b0']['per_source_accuracy'][s]['accuracy']*100:.1f}%",
    })
display(pd.DataFrame(src_rows))


## 6. Model Calibration & Reliability Diagrams (15 Equal-Width Bins)

Displaying calibration diagrams before and after temperature scaling on validation.


In [ ]:
for m, name in zip(models, names):
    print(f'=== {name} Reliability Diagram ===')
    display(IPImage(filename=str(ARTIFACTS_DIR / m / 'calibration.png')))


## 7. Explainability & Shortcut Learning Audit (Hand-Written Grad-CAM)

Inspecting convolutional feature attention using manual forward/backward PyTorch hooks.


### 7.1 Twelve-Class Correct Prediction Heatmaps
Verifying that the model activates on salient item geometry across all 12 classes.


In [ ]:
display(IPImage(filename=str(FIGURES_DIR / 'gradcam_12class_correct.png')))


### 7.2 Top 12 Confident Misclassifications with Grad-CAM
Inspecting the most confident failure cases and hazardous miss errors.

> **Key Observation:** In the top confident misclassifications (`gradcam_top_wrong.png`), note the `paper → cardboard` misclassification. The Grad-CAM heatmap intensity is concentrated in the bottom-right corner where a **shutterstock.com watermark** is printed, rather than on the paper item.


In [ ]:
display(IPImage(filename=str(FIGURES_DIR / 'gradcam_top_wrong.png')))


### 7.3 Shortcut & Watermark Bias Audit (TrashBox Test Samples)
Empirical check across 12 TrashBox test images (2 per class across 6 classes: `cardboard`, `e_waste`, `glass`, `medicine_syringe`, `metal`, `plastic`):
Does the model focus on the physical object or latch onto web watermarks, borders, and corners?

> **Visual Findings:** While central item geometry triggers high gradient energy in cases like syringes and circuit boards, several images display prominent heat on border edges (e.g. mobile phone left edge) and image corners (e.g. metal item corner), confirming partial reliance on border/watermark context.


In [ ]:
display(IPImage(filename=str(FIGURES_DIR / 'gradcam_shortcut_check.png')))


### 7.4 Quantitative Shortcut & Spatial Bias Evaluation

To quantitatively measure shortcut reliance and spatial bias beyond visual spot-checks, we evaluate all 2,843 test images under:
1. **Border-60% Accuracy:** Central 60% per side masked (36% area hidden, 64% border visible). Measures robustness to partial views + context, confounded by object parts in border.
2. **Border-10% Control:** Central 80% per side masked (64% area hidden, only a 10% perimeter frame remains visible). Isolates thin border context.
3. **Center-60% Accuracy:** Outer 20% border on all sides masked with dataset mean colour (only central 36% area visible).
4. **Grad-CAM Border Mass Ratio:** The outer 15% perimeter covers $1 - 0.70^2 = 51.0\%$ of image area. We compare observed border mass against this 51.0% uniform baseline. Ratios $< 1.0$ indicate that attention is on average **more center-focused** on the object than uniform chance.


In [ ]:
shortcut_json_path = REPORTS_DIR / 'shortcut_test.json'
if shortcut_json_path.exists():
    with open(shortcut_json_path) as f:
        sdata = json.load(f)

    s_rows = []
    for m, name in zip(models, names):
        if m in sdata:
            ov = sdata[m]['overall']
            b10_str = f"{ov['border_10_acc']*100:.2f}%" if 'border_10_acc' in ov else 'N/A'
            ratio_val = ov.get('gradcam_border_ratio', ov['gradcam_border_mass_mean']/0.51)
            s_rows.append({
                'Model': name,
                'Original Acc': f"{ov['original_acc']*100:.2f}%",
                'Center-60% Acc': f"{ov['center_60_acc']*100:.2f}%",
                'Border-60% Acc': f"{ov['border_60_acc']*100:.2f}%",
                'Border-10% Frame': b10_str,
                'Grad-CAM Border Mass': f"{ov['gradcam_border_mass_mean']*100:.1f}% ± {ov['gradcam_border_mass_std']*100:.1f}%",
                'Ratio vs 51% Uniform': f"{ratio_val:.2f}x ({'Center-Focused' if ratio_val < 1.0 else 'Border-Biased'})"
            })
    display(pd.DataFrame(s_rows))
else:
    print('shortcut_test.json not yet generated.')


## 8. Honest Academic Discussion & Engineering Findings

### 8.1 Key Findings
1. **Transfer Learning vs. From-Scratch:** Pretrained ImageNet representations provide a decisive advantage (+16.57 percentage points in accuracy, +0.1740 in macro-F1 over WasteNet). McNemar's test confirms this difference is overwhelmingly statistically significant ($p = 9.51 \times 10^{-91}$).

2. **Serving Model Selection:** MobileNetV3-Large was pre-selected per Decision **D-16** based on validation macro-F1 (0.922 vs 0.906) and its 2.1× CPU inference speed advantage (54.5 ms vs 115.4 ms). On the test set, MobileNetV3 achieves 92.44% accuracy vs 91.45% for EfficientNet-B0. A paired McNemar test reveals **$p = 0.0547$**, which is **not statistically significant** at $\alpha = 0.05$. Therefore, the serving decision rests appropriately on validation performance and computational efficiency rather than post-hoc test tuning.

3. **Hazard Miss Rate & Stream Routing:** Under SWM Rules 2026, hazardous materials (`sanitary` and `special_care`) must not leak into `wet` or `dry` municipal streams. MobileNetV3 and EfficientNet-B0 reduce hazard leakage to **3.12%** (21/674), down from 17.36% (117/674) in WasteNet. Overall stream-level classification accuracy reaches **97.82%**.

4. **Shortcut Learning & Attention Distribution:**
   - **Border Test Nuance:** Blanking the central 60% per side covers only 36% of the image area, leaving 64% visible. Because physical waste items often extend beyond this central box, the 82% border-60% accuracy partly reflects partial object visibility and context rather than pure shortcut reliance. The cleaner **Border-10% control** (masking the central 80% per side) provides a strict perimeter assessment.
   - **Grad-CAM Border Mass vs. Uniform Baseline:** The outer 15% perimeter covers $1 - 0.70^2 = 51.0\%$ of the image. Transfer learning models exhibit an average border mass of $\approx 42\%$, which is *below* uniform chance (ratio $\approx 0.83$). Thus, on average, model attention is **mostly object-centered**, accompanied by isolated individual cases of watermark or border reliance (e.g., the `shutterstock.com` watermark corner activation in `paper → cardboard`).

5. **Error Audit: Hazard Misses vs. Ambiguous Annotations:**
   - **Model Hazard Miss:** The confident misclassification of a copper wire coil as `metal` rather than `e_waste` is a genuine model hazard miss. TrashBox explicitly defines e-waste to include electrical wires, cords, and cables (Special Care stream); misrouting it to dry metal is a model classification failure under SWM Rules 2026.
   - **Ambiguous Annotations:** Other confident errors, such as transparent jars, reflect visual ambiguity between glass and clear rigid plastic in low-resolution web imagery.

6. **Confidence Thresholding Limitation:** The optimal uncertainty rejection threshold sits at the clamp floor ($p_{max} \ge 0.40$), retaining 96.0% coverage while elevating kept accuracy to 94.61%. This reflects that temperature-scaled probabilities remain relatively sharp after label smoothing.
